# Delta Lake com Apache Spark (PySpark)

Demonstração de **DDL, INSERT, UPDATE e DELETE** em uma tabela Delta.

## Cenário

Tabela única **`carros`**: o cadastro de veículos de uma seguradora. O dataset (`data/carros.csv`, com 8 registros) foi criado pelo grupo a partir do exemplo visto em aula.

### Modelo ER

```mermaid
erDiagram
    CARROS {
        string placa PK
        string modelo
        string chassi
        string marca
        int ano
        string cor
    }
```

| Coluna | Tipo | Descrição |
|---|---|---|
| placa | STRING | Placa do veículo (chave) |
| modelo | STRING | Modelo |
| chassi | STRING | Número do chassi |
| marca | STRING | Fabricante |
| ano | INT | Ano de fabricação |
| cor | STRING | Cor |

## 1. SparkSession com Delta Lake

`configure_spark_with_delta_pip` adiciona automaticamente o JAR do Delta compatível com a versão instalada pelo pip.

In [1]:
from pathlib import Path

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder.appName("delta-lake")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Pasta onde a tabela Delta será gravada (recriada a cada execução)
CAMINHO = (Path("..") / "warehouse" / "delta" / "carros").resolve()
spark.sql("DROP TABLE IF EXISTS carros")
if CAMINHO.exists():
    import shutil

    shutil.rmtree(CAMINHO)

spark.version

26/10/03 18:42:51 WARN Utils: Your hostname, NF-ENG-ISABELLEFELTRIN resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/03 18:42:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/isafel/trabalho-apachespark-eng_dados/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/isafel/.ivy2/cache
The jars for the packages stored in: /home/isafel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b5a09b6d-5688-485e-9b85-9a24cb1b6209;1.0
	confs: [default]


	found io.delta#delta-spark_2.12;3.3.2 in central


	found io.delta#delta-storage;3.3.2 in central


...

	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.3.2!delta-spark_2.12.jar (2054ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.3.2/delta-storage-3.3.2.jar ...


	[SUCCESSFUL ] io.delta#delta-storage;3.3.2!delta-storage.jar (229ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...


	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (295ms)
:: resolution report :: resolve 3568ms :: artifacts dl 2586ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   3   |   3   |   0   ||   3   |   3   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-b5a09b6d-5688-485e-9b85-9a24cb1b6209
	confs: [default]
	3 artifacts copied, 0 already retrieved (7313kB/13ms)


26/10/03 18:43:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


'3.5.9'

## Dados de origem (CSV)

In [2]:
df = spark.read.csv(
    "../data/carros.csv",
    header=True,
    schema="placa STRING, modelo STRING, chassi STRING, marca STRING, ano INT, cor STRING",
)
df.createOrReplaceTempView("carros_csv")
df.show()

+-------+---------+-----------+----------+----+-------+
|  placa|   modelo|     chassi|     marca| ano|    cor|
+-------+---------+-----------+----------+----+-------+
|ALD3834|     CLIO|34574215969|   RENAULT|2011| BRANCO|
|CCR8096|    CRETA|88547875547|   HYUNDAI|2020| BRANCO|
|DLA3438|    PUNTO|98823483434|      FIAT|2013|  PRETO|
|EEE1056|ECO SPORT|56753453455|      FORD|2020|   AZUL|
|FFR1234|    PALIO|32383478747|      FIAT|2020|AMARELO|
|GQY6753|      S10|72004160549|        GM|2015|  PRETO|
|IAC8974|   TIGUAN|77130757746|VOLKSWAGEN|2020|   AZUL|
|JIE0952|   PASSAT|87493270405|VOLKSWAGEN|2016|  CINZA|
+-------+---------+-----------+----------+----+-------+



## 2. DDL: criação da tabela Delta

In [3]:
spark.sql(f"""
    CREATE TABLE carros (
        placa  STRING,
        modelo STRING,
        chassi STRING,
        marca  STRING,
        ano    INT,
        cor    STRING
    )
    USING delta
    LOCATION '{CAMINHO}'
""")

spark.sql("INSERT INTO carros SELECT * FROM carros_csv")  # carga inicial
spark.sql("SELECT * FROM carros ORDER BY placa").show()

26/10/03 18:43:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+---------+-----------+----------+----+-------+
|  placa|   modelo|     chassi|     marca| ano|    cor|
+-------+---------+-----------+----------+----+-------+
|ALD3834|     CLIO|34574215969|   RENAULT|2011| BRANCO|
|CCR8096|    CRETA|88547875547|   HYUNDAI|2020| BRANCO|
|DLA3438|    PUNTO|98823483434|      FIAT|2013|  PRETO|
|EEE1056|ECO SPORT|56753453455|      FORD|2020|   AZUL|
|FFR1234|    PALIO|32383478747|      FIAT|2020|AMARELO|
|GQY6753|      S10|72004160549|        GM|2015|  PRETO|
|IAC8974|   TIGUAN|77130757746|VOLKSWAGEN|2020|   AZUL|
|JIE0952|   PASSAT|87493270405|VOLKSWAGEN|2016|  CINZA|
+-------+---------+-----------+----------+----+-------+



## 3. INSERT

Inclusão de um carro novo.

In [4]:
spark.sql("""
    INSERT INTO carros VALUES ('KLM2468', 'ONIX', '11223344556', 'GM', 2023, 'PRATA')
""")
spark.sql("SELECT * FROM carros ORDER BY placa").show()

+-------+---------+-----------+----------+----+-------+
|  placa|   modelo|     chassi|     marca| ano|    cor|
+-------+---------+-----------+----------+----+-------+
|ALD3834|     CLIO|34574215969|   RENAULT|2011| BRANCO|
|CCR8096|    CRETA|88547875547|   HYUNDAI|2020| BRANCO|
|DLA3438|    PUNTO|98823483434|      FIAT|2013|  PRETO|
|EEE1056|ECO SPORT|56753453455|      FORD|2020|   AZUL|
|FFR1234|    PALIO|32383478747|      FIAT|2020|AMARELO|
|GQY6753|      S10|72004160549|        GM|2015|  PRETO|
|IAC8974|   TIGUAN|77130757746|VOLKSWAGEN|2020|   AZUL|
|JIE0952|   PASSAT|87493270405|VOLKSWAGEN|2016|  CINZA|
|KLM2468|     ONIX|11223344556|        GM|2023|  PRATA|
+-------+---------+-----------+----------+----+-------+



## 4. UPDATE

O ECO SPORT (placa `EEE1056`) foi pintado de vermelho.

In [5]:
spark.sql("UPDATE carros SET cor = 'VERMELHO' WHERE placa = 'EEE1056'")
spark.sql("SELECT * FROM carros WHERE placa = 'EEE1056'").show()

+-------+---------+-----------+-----+----+--------+
|  placa|   modelo|     chassi|marca| ano|     cor|
+-------+---------+-----------+-----+----+--------+
|EEE1056|ECO SPORT|56753453455| FORD|2020|VERMELHO|
+-------+---------+-----------+-----+----+--------+



## 5. DELETE

Exclusão dos carros fabricados antes de 2015 (CLIO 2011 e PUNTO 2013).

In [6]:
spark.sql("DELETE FROM carros WHERE ano < 2015")
spark.sql("SELECT * FROM carros ORDER BY placa").show()

+-------+---------+-----------+----------+----+--------+
|  placa|   modelo|     chassi|     marca| ano|     cor|
+-------+---------+-----------+----------+----+--------+
|CCR8096|    CRETA|88547875547|   HYUNDAI|2020|  BRANCO|
|EEE1056|ECO SPORT|56753453455|      FORD|2020|VERMELHO|
|FFR1234|    PALIO|32383478747|      FIAT|2020| AMARELO|
|GQY6753|      S10|72004160549|        GM|2015|   PRETO|
|IAC8974|   TIGUAN|77130757746|VOLKSWAGEN|2020|    AZUL|
|JIE0952|   PASSAT|87493270405|VOLKSWAGEN|2016|   CINZA|
|KLM2468|     ONIX|11223344556|        GM|2023|   PRATA|
+-------+---------+-----------+----------+----+--------+



## 6. Histórico e time travel

Cada operação gera uma nova **versão** da tabela no `_delta_log`.

In [7]:
spark.sql("DESCRIBE HISTORY carros").select("version", "timestamp", "operation").show(truncate=False)

+-------+-----------------------+------------+
|version|timestamp              |operation   |
+-------+-----------------------+------------+
|4      |2026-10-03 18:43:26.467|DELETE      |
|3      |2026-10-03 18:43:23.255|UPDATE      |
|2      |2026-10-03 18:43:20.467|WRITE       |
|1      |2026-10-03 18:43:16.315|WRITE       |
|0      |2026-10-03 18:43:13.947|CREATE TABLE|
+-------+-----------------------+------------+



Consultando a tabela como ela estava **logo após a carga inicial** (versão 1):

In [8]:
spark.sql("SELECT * FROM carros VERSION AS OF 1 ORDER BY placa").show()

+-------+---------+-----------+----------+----+-------+
|  placa|   modelo|     chassi|     marca| ano|    cor|
+-------+---------+-----------+----------+----+-------+
|ALD3834|     CLIO|34574215969|   RENAULT|2011| BRANCO|
|CCR8096|    CRETA|88547875547|   HYUNDAI|2020| BRANCO|
|DLA3438|    PUNTO|98823483434|      FIAT|2013|  PRETO|
|EEE1056|ECO SPORT|56753453455|      FORD|2020|   AZUL|
|FFR1234|    PALIO|32383478747|      FIAT|2020|AMARELO|
|GQY6753|      S10|72004160549|        GM|2015|  PRETO|
|IAC8974|   TIGUAN|77130757746|VOLKSWAGEN|2020|   AZUL|
|JIE0952|   PASSAT|87493270405|VOLKSWAGEN|2016|  CINZA|
+-------+---------+-----------+----------+----+-------+



## 7. Arquivos gerados

Dados em **Parquet** + log de transações em **JSON** (`_delta_log`).

In [9]:
for arquivo in sorted(CAMINHO.rglob("*")):
    if arquivo.is_file() and not arquivo.name.endswith(".crc"):
        print(arquivo.relative_to(CAMINHO))

_delta_log/00000000000000000000.json
_delta_log/00000000000000000001.json
_delta_log/00000000000000000002.json
_delta_log/00000000000000000003.json
_delta_log/00000000000000000004.json
part-00000-220c37e7-ba5c-4329-a834-35ddce6ff333-c000.snappy.parquet
part-00000-3f8f07b2-77c6-4bd3-ae1f-214a6500494a-c000.snappy.parquet
part-00000-866bd674-02fc-4bc4-9efd-c7f4343dc9c1-c000.snappy.parquet
part-00000-f1762007-fbe6-42d9-b3fa-1284e6506b6b-c000.snappy.parquet


In [10]:
spark.stop()